> **Superseded:** this notebook trains the older single-head (flat 6-class) model. For the current multi-head model (health category + condition + 0-10 health score + confidence calibration), use [`MantisVision_Training_MultiHead.ipynb`](./MantisVision_Training_MultiHead.ipynb) instead. Checkpoints produced by this notebook are not compatible with the current `ml/src` inference pipeline.

# Mantis Vision — Train in Google Colab

Run these cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

This notebook: clones the repo, installs dependencies, lets you upload your labeled photo dataset directly, trains the EfficientNet-B0 health classifier, evaluates it, and lets you download the resulting checkpoint.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Arran0/MantisVision.git
%cd MantisVision/ml

## 2. Install dependencies

Colab already has torch/torchvision preinstalled with GPU support, so this just adds the extra packages the pipeline needs.

In [ ]:
!pip install -q grad-cam fastapi python-multipart

## 3. Upload your dataset

On your own computer, organize your photos into `train/validation/test/<Class>/` folders (see `docs/DATASET_LABELING_GUIDE.md`) — each split (`train`, `validation`, `test`) contains one subfolder per class (`Healthy`, `Moderate`, etc.) full of images. Keep the class subfolders intact; don't flatten them out.

Then select the **`train`, `validation`, and `test` folders themselves** (all three, as siblings) and zip them together into one `.zip`. Do **not** zip only the loose images, and do **not** zip a parent folder that wraps around `train`/`validation`/`test` — when someone unzips your file, `train/`, `validation/`, and `test/` should appear immediately, not nested inside another folder. (If you do end up with an extra wrapper level anyway, or a class folder is called `val`/`valid`/`validate` instead of `validation`, or the zip carries macOS `__MACOSX`/`.DS_Store` clutter, the cell below auto-detects and fixes all of that.)

Run the cell below, then use the file picker to select that `.zip`.

In [ ]:
import pathlib
import shutil
import zipfile

from google.colab import files

dataset_dir = pathlib.Path("dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

print("Select the zip file containing your train/validation/test folders...")
uploaded = files.upload()

for name in uploaded:
    if name.lower().endswith(".zip"):
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(name) as zf:
            zf.extractall(dataset_dir)
    else:
        print(f"Skipping non-zip file: {name} (expected a single .zip upload)")

# Clean up macOS zip artifacts: the __MACOSX/ sidecar folder plus .DS_Store
# and AppleDouble "._*" files that macOS's Archive Utility sprinkles into
# zips made on a Mac. None of these are real images, so they'd otherwise
# just sit alongside the class folders as clutter.
shutil.rmtree(dataset_dir / "__MACOSX", ignore_errors=True)
for junk in list(dataset_dir.rglob(".DS_Store")) + list(dataset_dir.rglob("._*")):
    if junk.is_file():
        junk.unlink()

# Auto-fix: find wherever train/validation/test actually ended up, however
# deep, and however "validation" was spelled (val/valid/validate are all
# accepted), then move+rename them into dataset/{train,validation,test}.
# Handles zips that wrap everything in a parent folder (e.g. the zip's name)
# as well as folders named e.g. "validate" instead of "validation". The repo
# already ships empty dataset/{train,validation,test}/<Class>/.gitkeep
# placeholders, so we can't just check "do these three folders exist" — we
# specifically look for the shallowest match that actually contains images.
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}
split_aliases = {
    "train": {"train", "training"},
    "validation": {"validation", "val", "valid", "validate"},
    "test": {"test", "testing"},
}


def canonical_split(dirname):
    lowered = dirname.lower()
    for canonical, aliases in split_aliases.items():
        if lowered in aliases:
            return canonical
    return None


def has_images(path):
    return any(f.is_file() and f.suffix.lower() in IMAGE_EXTS for f in path.rglob("*"))


candidates = sorted(
    [dataset_dir] + [p for p in dataset_dir.rglob("*") if p.is_dir()],
    key=lambda p: len(p.relative_to(dataset_dir).parts),
)

holder, matches = None, {}
for candidate in candidates:
    found = {}
    for child in candidate.iterdir():
        if not child.is_dir():
            continue
        canonical = canonical_split(child.name)
        if canonical and canonical not in found:
            found[canonical] = child
    if {"train", "validation", "test"}.issubset(found) and any(has_images(d) for d in found.values()):
        holder, matches = candidate, found
        break

if holder is not None:
    if holder != dataset_dir:
        print(f"Found the splits inside '{holder.relative_to(dataset_dir)}/', moving them up...")
    for canonical, child in matches.items():
        dest = dataset_dir / canonical
        if child.resolve() == dest.resolve():
            continue
        if child.name != canonical:
            print(f"Renaming '{child.name}/' -> '{canonical}/'")
        if dest.exists():
            shutil.copytree(child, dest, dirs_exist_ok=True)
            shutil.rmtree(child)
        else:
            shutil.move(str(child), str(dest))
    if holder != dataset_dir:
        shutil.rmtree(holder, ignore_errors=True)
else:
    print("Warning: could not find any images under train/validation/test folders in the zip.")


def print_tree(path, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    for entry in sorted(path.iterdir()):
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir():
            print_tree(entry, prefix + "  ", depth + 1, max_depth)


print(f"\nContents of {dataset_dir.resolve()}:\n")
print_tree(dataset_dir)

## 4. Validate the dataset

Checks every class folder exists, every image opens correctly, and flags underrepresented classes.

In [ ]:
!python -m src.data.validate_dataset

## 5. Train

Two-phase schedule (frozen backbone, then fine-tune) with early stopping. Saves the best checkpoint to `ml/checkpoints/best_model.pt`. This can take a while depending on dataset size — Colab's free GPU tier helps a lot here.

In [ ]:
!python -m src.train

## 6. Evaluate

Accuracy/precision/recall/F1 (macro + per-class), confusion matrix, ROC AUC on the held-out test split.

In [ ]:
!python -m src.evaluate

from IPython.display import Image, display
display(Image(filename="reports/confusion_matrix.png"))

## 7. Download the trained checkpoint

Colab runtimes are ephemeral — grab the checkpoint now so you don't lose it when the session disconnects. You'll need this file to run the inference API later.

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pt")

## Optional: try a Grad-CAM explanation on one photo

Upload a single test photo and see the heatmap of what the model looked at.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick one photo
photo_path = list(uploaded.keys())[0]
!python -m src.gradcam "{photo_path}"

from IPython.display import Image, display
import pathlib
display(Image(filename=str(pathlib.Path(photo_path).with_suffix('.gradcam.png'))))